In [1]:
!nvidia-smi

Mon Nov  3 15:18:41 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q torch torchvision torchaudio
!pip install -q --upgrade transformers datasets peft bitsandbytes accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [3]:
import os, shutil, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
WORKDIR = "/content/pattern_miner"
ADAPTER_DIR = f"{WORKDIR}/adapters"
MERGED_DIR = f"{WORKDIR}/merged"
EXPORTS = f"{WORKDIR}/exports"
for d in [ADAPTER_DIR, MERGED_DIR, EXPORTS]: os.makedirs(d, exist_ok=True)

print("Loading dataset...")
dataset = load_dataset("json", data_files="/content/dataset.json", split="train")
print("Loaded", len(dataset), "examples")

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb_cfg, device_map="auto", trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 60 examples


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

def format_chat(examples):
    texts = []
    for msgs in examples["messages"]:
        t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(t)
    return {"text": texts}

formatted = dataset.map(format_chat, batched=True)

def tokenize_fn(examples):
    enc = tokenizer(examples["text"], truncation=True, max_length=1024, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized = formatted.map(tokenize_fn, batched=True, remove_columns=formatted.column_names)

print("Training starting...")
targs = TrainingArguments(
    output_dir=f"{WORKDIR}/outputs",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    fp16=torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    save_total_limit=3,
    report_to="none",
)

trainable params: 6,307,840 || all params: 1,106,356,224 || trainable%: 0.5701


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Training starting...


In [5]:
trainer = Trainer(model=model, args=targs, train_dataset=tokenized)
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.948200
20,1.704400


TrainOutput(global_step=24, training_loss=1.769752522309621, metrics={'train_runtime': 245.4108, 'train_samples_per_second': 0.733, 'train_steps_per_second': 0.098, 'total_flos': 1151063902126080.0, 'train_loss': 1.769752522309621, 'epoch': 3.0})

In [8]:
print("Saving adapter...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

Saving adapter...


('/content/pattern_miner/adapters/tokenizer_config.json',
 '/content/pattern_miner/adapters/special_tokens_map.json',
 '/content/pattern_miner/adapters/chat_template.jinja',
 '/content/pattern_miner/adapters/tokenizer.model',
 '/content/pattern_miner/adapters/added_tokens.json',
 '/content/pattern_miner/adapters/tokenizer.json')

In [9]:
print("Merging LoRA with base model...")
merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

Merging LoRA with base model...


('/content/pattern_miner/merged/tokenizer_config.json',
 '/content/pattern_miner/merged/special_tokens_map.json',
 '/content/pattern_miner/merged/chat_template.jinja',
 '/content/pattern_miner/merged/tokenizer.model',
 '/content/pattern_miner/merged/added_tokens.json',
 '/content/pattern_miner/merged/tokenizer.json')

In [ ]:
shutil.make_archive(f"{EXPORTS}/adapters", "zip", ADAPTER_DIR)
shutil.make_archive(f"{EXPORTS}/merged", "zip", MERGED_DIR)
print("Done! Files saved to", EXPORTS)

In [10]:
from google.colab import drive
import shutil, os

# Source and destination
src = "/content/pattern_miner"
dst = "/content/drive/MyDrive/pattern_miner"

# Remove old copy (optional)
if os.path.exists(dst):
    shutil.rmtree(dst)

# Copy the folder
shutil.copytree(src, dst)

print(f"Model folder moved to: {dst}")

Model folder moved to: /content/drive/MyDrive/pattern_miner
